# 🌹 Rose Python SDK - Complete Demo

Welcome to the comprehensive Rose Python SDK demonstration! This notebook showcases the full capabilities of Rose's recommendation system through a practical, step-by-step tutorial.

## 🎯 What You'll Learn

This demo covers the complete Rose workflow through **5 key steps**:

1. **📊 Create Datasets** - Set up interaction and metadata datasets with proper schemas
2. **🔧 Create Pipeline** - Build a realtime-leaderboard recommendation pipeline
3. **⚡ Record API Ingestion** - Ingest individual records using the Record API
4. **📦 Batch Append** - Add records to existing datasets using batch-append
5. **🔄 Batch Overwrite** - Replace entire datasets using batch-overwrite

## 🏗️ Architecture Overview

```
┌─────────────────┐    ┌─────────────────┐    ┌──────────────────┐
│   Interaction   │    │    Metadata     │    │    Pipeline      │
│    Dataset      │    │    Dataset      │    │  (Algorithm)     │
│                 │    │                 │    │                  │
│ • user_id       │    │ • item_id       │    │ • realtime_      │
│ • item_id       │    │ • title         │    │   leaderboard    │
│ • interaction   │    │ • genre         │    │ • personalized   │
│ • timestamp     │    │ • description   │    │   recommendations│
└─────────────────┘    └─────────────────┘    └──────────────────┘
         │                       │                       │
         └───────────────────────┼───────────────────────┘
                                 │
                    ┌─────────────────┐
                    │  Rose Engine    │
                    │                 │
                    │ • ML Processing │
                    │ • Real-time     │
                    │ • Scalable      │
                    └─────────────────┘
```

## 🚀 Prerequisites

- Python 3.8+
- Rose Python SDK installed
- Valid Rose API credentials
- Basic understanding of recommendation systems

Let's get started! 🎉

```


## 📦 Installation & Setup

First, let's install the Rose Python SDK in development mode to ensure we have the latest version:

In [2]:
# Install Rose Python SDK in development mode
%pip uninstall rose-python-sdk -y
%pip install -e .

Found existing installation: rose-python-sdk 0.0.0
Uninstalling rose-python-sdk-0.0.0:
  Successfully uninstalled rose-python-sdk-0.0.0
Note: you may need to restart the kernel to use updated packages.
Obtaining file:///Users/lulilee/KKS/RaaS/rose-python-sdk
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rose-python-sdk (pyproject.toml) ... done
  Created wheel for rose-python-sdk: filename=rose_python_sdk-0.0.0-0.editable-py3-none-any.whl size=7122 sha256=0b7e3af9970f234a8d0ea1399123972327d4bf0661f606c6ccbc6806b73e9860
  Stored in directory: /private/var/folders/c_/qwqzg5dn08xfcv7gkvj9vyt40000gp/T/pip-ephem-wheel-cache-1nvhf708/wheels/ec/63/28/ebf0742ab28b99ca897bbf7460d8ba04b4ee1c0d4e05cd73cc
Successfully built rose-python-sdk

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update

## 📚 Import Required Libraries

Let's import all the necessary libraries and Rose SDK components:


In [1]:
# Standard library imports
import os
import time
import json
import pprint
from typing import List, Dict, Any

# Rose SDK imports
from rose_sdk import RoseClient
from rose_sdk.helpers import (
    build_schema_from_sample,
    convert_records_to_rose_format,
    quick_create_dataset_with_data
)
from rose_sdk.utils.pipeline import create_pipeline
from rose_sdk.exceptions import RoseAPIError

print("✅ All imports successful!")


✅ All imports successful!


/Users/lulilee/KKS/RaaS/rose-python-sdk/demo_env/lib/python3.11/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "CreateDatasetRequest" shadows an attribute in parent "BaseModel"
  warnings.warn(
/Users/lulilee/KKS/RaaS/rose-python-sdk/demo_env/lib/python3.11/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "Dataset" shadows an attribute in parent "BaseModel"
  warnings.warn(


## 🔐 Initialize Rose Client

Connect to the Rose API using your credentials:


In [2]:
# Configure Rose API connection
BASE_URL = os.getenv('ROSE_BASE_URL', 'https://api.rose.kkv-test.com')
ACCESS_TOKEN = 'u2ww3qaHjB+ELK/1+d4JqHKKRu0HYYPYUpW+D9qa2w0='

# Initialize Rose client
client = RoseClient(
    base_url=BASE_URL,
    access_token=ACCESS_TOKEN
)

print(f"🔗 Connected to Rose API at: {BASE_URL}")
print("✅ Rose client initialized successfully!")


🔗 Connected to Rose API at: https://api.rose.kkv-test.com
✅ Rose client initialized successfully!


In [3]:
client.health_check()

## ⚙️ Configuration

Set up the names for our datasets and pipeline:

In [4]:
# Dataset and pipeline names
interaction_dataset_name = "internal_interaction"
metadata_dataset_name = "internal_metadata"
demo_pipeline_name = "internal_pipeline"

print("📋 Configuration set:")
print(f"   Interaction Dataset: {interaction_dataset_name}")
print(f"   Metadata Dataset: {metadata_dataset_name}")
print(f"   Pipeline Name: {demo_pipeline_name}")


📋 Configuration set:
   Interaction Dataset: internal_interaction
   Metadata Dataset: internal_metadata
   Pipeline Name: internal_pipeline


# 📊 Step 1: Create Datasets

In this step, we'll create two essential datasets that form the foundation of our recommendation system:

## 🎯 Dataset Overview

### 📈 Interaction Dataset
**Purpose**: Captures user behavior and preferences through their interactions with items.

**Key Fields**:
- `user_id` - Unique identifier for each user
- `item_id` - Unique identifier for each item
- `interaction_type` - Type of interaction (play, click, like, share, etc.)
- `timestamp` - When the interaction occurred
- `rating` - User's rating or feedback (optional)
- `play_duration` - How long the user engaged (for media items)

**Use Cases**:
- Track user preferences and behavior patterns
- Identify popular items and trending content
- Build user profiles for personalized recommendations

### 📋 Metadata Dataset
**Purpose**: Stores descriptive information about items to provide context for recommendations.

**Key Fields**:
- `item_id` - Unique identifier for each item
- `title` - Item name or title
- `item_type` - Category of item (music, video, article, etc.)
- `genre` - Content genre or category
- `description` - Detailed description of the item
- `artist` - Creator or artist name
- `duration` - Length of the item (for media)

**Use Cases**:
- Provide rich context for recommendation algorithms
- Enable content-based filtering
- Support faceted search and filtering

---

Let's create these datasets step by step! 🚀


## 🔧 Create Interaction Dataset Schema

First, let's define the schema for our interaction dataset using sample data:

In [5]:
# Create interaction dataset schema
print("🔹 CREATING INTERACTION DATASET SCHEMA")
print("=" * 50)
import time
# Sample interaction records to build schema from
interaction_sample_records = [
    {
        "user_id": "user_001",
        "item_id": "item_001", 
        "item_type": "video", 
        "interaction_type": "play",
        "play_duration": 180,
        "timestamp": int(time.time()),
        "rating": 4.5
    }
]

# Build schema from sample records
interaction_schema = build_schema_from_sample(
    sample_records=interaction_sample_records,
    identifier_fields=["user_id", "item_id", "item_type", "timestamp"],
    required_fields=["user_id", "item_id", "item_type", "interaction_type", "timestamp"]
)

print("📋 Interaction Dataset Schema:")
for field_name, field in interaction_schema.items():
    print(f"   {field_name}: {field.field_type} (Identifier: {field.is_identifier}, Required: {field.is_required})")


🔹 CREATING INTERACTION DATASET SCHEMA
📋 Interaction Dataset Schema:
   item_type: str (Identifier: True, Required: True)
   play_duration: int (Identifier: False, Required: False)
   rating: float (Identifier: False, Required: False)
   interaction_type: str (Identifier: False, Required: True)
   user_id: str (Identifier: True, Required: True)
   timestamp: int (Identifier: True, Required: True)
   item_id: str (Identifier: True, Required: True)


## 🏗️ Create Interaction Dataset

Now let's create the interaction dataset using the schema we defined:

In [6]:
# Create the interaction dataset
try:
    interaction_dataset = client.datasets.create(
        name=interaction_dataset_name,
        schema=interaction_schema,
        enable_housekeeping=True
    )
    
    interaction_dataset_id = interaction_dataset.dataset_id
    print(f"✅ Interaction dataset created successfully!")
    print(f"   Dataset ID: {interaction_dataset_id}")
    print(f"   Dataset Name: {interaction_dataset_name}")
    
except RoseAPIError as e:
    print(f"❌ Failed to create interaction dataset: {e.message}")
    # Try to find existing dataset
    datasets = client.datasets.list()
    for dataset in datasets:
        if dataset.dataset_name == interaction_dataset_name:
            interaction_dataset_id = dataset.dataset_id
            print(f"📋 Found existing interaction dataset: {interaction_dataset_id}")
            break
    else:
        raise Exception("Could not create or find interaction dataset")


❌ Failed to create interaction dataset: conflict: [DATA_CONFLICT] dataset already exists
📋 Found existing interaction dataset: ce-yQfXKSmq4TUu6pJE9OA


## 🔧 Create Metadata Dataset Schema

Now let's create the metadata dataset schema with sample item information:


In [7]:
# Create metadata dataset schema
print("\n🔹 CREATING METADATA DATASET SCHEMA")
print("=" * 50)

# Sample metadata records to build schema from
metadata_sample_records = [
    {
        "item_id": "item_001",
        "title": "Amazing Song",
        "item_type": "Music",
        "description": "A beautiful melody that touches the heart",
        "artist": "Artist One",
        "genres": ["Pop","Comedy"],
        "duration": 180,
        "release_date": "2024-01-01T00:00:00Z"
    }
]

# Build schema from sample records
metadata_schema = build_schema_from_sample(
    sample_records=metadata_sample_records,
    identifier_fields=["item_id", "item_type"],
    required_fields=["item_id", "title", "item_type", "genres"]
)

print("📋 Metadata Dataset Schema:")
for field_name, field in metadata_schema.items():
    print(f"   {field_name}: {field.field_type} (Identifier: {field.is_identifier}, Required: {field.is_required})")



🔹 CREATING METADATA DATASET SCHEMA
📋 Metadata Dataset Schema:
   release_date: str (Identifier: False, Required: False)
   item_type: str (Identifier: True, Required: True)
   description: str (Identifier: False, Required: False)
   artist: str (Identifier: False, Required: False)
   duration: int (Identifier: False, Required: False)
   title: str (Identifier: False, Required: True)
   genres: list (Identifier: False, Required: True)
   item_id: str (Identifier: True, Required: True)


In [8]:
# Create the metadata dataset
try:
    metadata_dataset = client.datasets.create(
        name=metadata_dataset_name,
        schema=metadata_schema,
        enable_housekeeping=True
    )
    
    metadata_dataset_id = metadata_dataset.dataset_id
    print(f"✅ Metadata dataset created successfully!")
    print(f"   Dataset ID: {metadata_dataset_id}")
    print(f"   Dataset Name: {metadata_dataset_name}")
    
except RoseAPIError as e:
    print(f"❌ Failed to create metadata dataset: {e.message}")
    # Try to find existing dataset
    datasets = client.datasets.list()
    for dataset in datasets:
        if dataset.dataset_name == metadata_dataset_name:
            metadata_dataset_id = dataset.dataset_id
            print(f"📋 Found existing metadata dataset: {metadata_dataset_id}")
            break
    else:
        raise Exception("Could not create or find metadata dataset")

print(f"\n🎉 Step 1 Complete! Created datasets:")
print(f"   📈 Interaction Dataset ID: {interaction_dataset_id}")
print(f"   📋 Metadata Dataset ID: {metadata_dataset_id}")


❌ Failed to create metadata dataset: conflict: [DATA_CONFLICT] dataset already exists
📋 Found existing metadata dataset: UWJ8Q0OoTmqn7NRBC3iZiQ

🎉 Step 1 Complete! Created datasets:
   📈 Interaction Dataset ID: ce-yQfXKSmq4TUu6pJE9OA
   📋 Metadata Dataset ID: UWJ8Q0OoTmqn7NRBC3iZiQ


# 🔧 Step 2: Create Recommendation Pipeline

Now that we have our datasets ready, let's create a recommendation pipeline that will process our data and generate personalized recommendations.

## 🎯 What is a Pipeline?

A **pipeline** in Rose is a configuration that defines how recommendation algorithms process your data to generate personalized recommendations. It acts as the bridge between your raw data and the intelligent recommendations your users will receive.

## 🏗️ Pipeline Components

### 1. **Dataset Mapping**
- Links your datasets to pipeline input requirements
- Maps interaction data to user behavior patterns
- Connects metadata to item context

### 2. **Scenario Configuration**
- Defines the recommendation algorithm and behavior
- Specifies how recommendations are generated
- Configures real-time vs batch processing

### 3. **Query Generation** (Automatic)
- Creates query endpoints for recommendations
- Enables real-time recommendation requests
- Provides RESTful API access

## 🎪 Realtime-Leaderboard Scenario

We'll use the **realtime-leaderboard** scenario, which is perfect for:
- **Real-time recommendations**: Instant response to user actions
- **Popular content**: Trending items and viral content
- **Personalized rankings**: User-specific content ordering
- **Dynamic updates**: Recommendations that change based on current activity

---

Let's create our pipeline! 🚀


## 🔧 Configure Pipeline

Let's set up our pipeline configuration with the realtime-leaderboard scenario:

In [9]:
# Configure pipeline
print("🔹 CREATING REALTIME-LEADERBOARD PIPELINE")
print("=" * 50)

# Define pipeline configuration
pipeline_name = "test-query-expr"
scenario = "realtime_leaderboard"

# Map our datasets to pipeline requirements
dataset_mapping = {
    "interaction": interaction_dataset_name,  # User-item interactions
    "metadata": metadata_dataset_name         # Item metadata
}

print("📋 Pipeline Configuration:")
print(f"   Pipeline Name: {pipeline_name}")
print(f"   Scenario: {scenario}")
print(f"   Dataset Mapping:")
print(f"     - interaction: {interaction_dataset_name}")
print(f"     - metadata: {metadata_dataset_name}")

# Create pipeline configuration using the helper function
pipeline_config = create_pipeline(
    pipeline_name=pipeline_name,
    scenario=scenario,
    dataset_mapping=dataset_mapping
)

print(f"\n📋 Generated Pipeline Configuration:")
print(f"   Pipeline Name: {pipeline_config['pipeline_name']}")
print(f"   Scenario: {pipeline_config['properties']['scenario']}")
print(f"   Datasets: {list(pipeline_config['properties']['datasets'].keys())}")


🔹 CREATING REALTIME-LEADERBOARD PIPELINE
📋 Pipeline Configuration:
   Pipeline Name: test-query-expr
   Scenario: realtime_leaderboard
   Dataset Mapping:
     - interaction: internal_interaction
     - metadata: internal_metadata

📋 Generated Pipeline Configuration:
   Pipeline Name: test-query-expr
   Scenario: realtime_leaderboard
   Datasets: ['interaction', 'metadata']


## 🏗️ Create Pipeline

Now let's create the pipeline using our configuration:

In [10]:
# Create the pipeline
try:
    pipeline_response = client.pipelines.create(
        name=pipeline_config["pipeline_name"],
        properties=pipeline_config["properties"]
    )
    
    pipeline_id = pipeline_response.pipeline_id
    
    print(f"\n✅ Pipeline created successfully!")
    print(f"   Pipeline ID: {pipeline_id}")
    
except RoseAPIError as e:
    print(f"❌ Failed to create pipeline: {e.message}")
    # Try to find existing pipeline
    pipelines = client.pipelines.list()
    for pipeline in pipelines:
        if pipeline.pipeline_name == pipeline_name:
            pipeline_id = pipeline.pipeline_id
            print(f"📋 Found existing pipeline: {pipeline_id}")
            break
    else:
        raise Exception("Could not create or find pipeline")

# Check pipeline status
pipeline_status = client.pipelines.get(pipeline_id)
print(f"   Pipeline Name: {pipeline_status.pipeline_name}")
print(f"   Status: {pipeline_status.status}")
print(f"   Properties: {pipeline_status.properties}")
print("=" * 50)
print(f"\n🎉 Step 2 Complete! Created pipeline:")
print(f"   Pipeline ID: {pipeline_id}")
print(f"   Pipeline Name: {pipeline_name}")
print(f"   Scenario: {scenario}")


❌ Failed to create pipeline: conflict: [DATA_CONFLICT] pipeline already exists
📋 Found existing pipeline: aAGYFsdwT5-17XKztWXPww
   Pipeline Name: test-query-expr
   Status: CREATE SUCCESSFUL
   Properties: {'datasets': {'interaction': 'internal_interaction', 'metadata': 'internal_metadata'}, 'scenario': 'realtime_leaderboard'}

🎉 Step 2 Complete! Created pipeline:
   Pipeline ID: aAGYFsdwT5-17XKztWXPww
   Pipeline Name: test-query-expr
   Scenario: realtime_leaderboard


## ⏳ Monitor Pipeline Status

Let's wait for the pipeline to be ready to serve recommendations:
        

In [11]:
# Monitor pipeline status until ready
from rose_sdk.models.pipeline import PipelineStatus
import time

print("⏳ Waiting for pipeline to be ready...")
while True:
    pipeline_status = client.pipelines.get(pipeline_id)
    print(f"   Pipeline Name: {pipeline_status.pipeline_name}")
    print(f"   Status: {pipeline_status.status}")
    
    if pipeline_status.status == PipelineStatus.CREATE_SUCCESSFUL:
        print(f"\n🎉 Pipeline {pipeline_status.pipeline_name} now ready to serve!")
        break
    elif pipeline_status.status == "waiting":
        print("   Pipeline still creating...")
        time.sleep(3)
    else:
        print(f"   ❌ Pipeline creation failed with status: {pipeline_status.status}")
        break


⏳ Waiting for pipeline to be ready...
   Pipeline Name: test-query-expr
   Status: CREATE SUCCESSFUL

🎉 Pipeline test-query-expr now ready to serve!


# ⚡ Step 3: Record API Ingestion

Now that our pipeline is ready, let's start ingesting data! We'll begin with the **Record API** - the most straightforward way to add individual records to your datasets.

## 🎯 What is Record API Ingestion?

The **Record API** allows you to add individual records or small batches of records directly to your datasets. It's perfect for:

### ✅ Use Cases
- **Real-time data streaming**: Adding records as they occur in your application
- **Small batch updates**: Adding a few records at a time
- **Interactive applications**: User actions, clicks, ratings, purchases
- **Testing and development**: Quick data insertion for demos and testing
- **Event-driven updates**: Responding to user interactions immediately

### 🔄 How It Works
1. **Format Data**: Convert your records to Rose's internal format
2. **Send Request**: POST records directly to the dataset
3. **Immediate Processing**: Records are processed and available for recommendations
4. **Real-time Updates**: Pipeline immediately incorporates new data

---

Let's ingest some sample interaction data! 🚀

## 🔧 Generate Sample Interaction Data

Let's create realistic sample data for our Record API ingestion:

In [12]:
# Create more diverse and realistic data
interaction_types = ["play", "click", "like", "share"]
item_types = ["music", "video"]
user_ids = ["luli", "wayne", "chloe", "yaru", "Yen", "Vis"]
item_ids = ["Wednesday (Season 1)" , "Adolescence (Limited Series)" , "Stranger Things 4" , "DAHMER: Monster: The Jeffrey Dahmer Story", "Bridgerton (Season 1)", "The Queen's Gambit (Limited Series)"]

In [13]:
# Generate sample interaction records
print("🔹 GENERATING SAMPLE INTERACTION DATA")
print("=" * 50)



# Generate 10 sample interaction records
from datetime import datetime, timedelta
import random

record_api_records = []

for i in range(15):
    record = {
        "user_id": user_ids[i % len(user_ids)],  # 5 unique users
        "item_id": item_ids[i % len(item_ids)],  # 8 unique items
        "item_type": item_types[random.randint(0, len(item_types))-1],
        "interaction_type": interaction_types[i % len(interaction_types)],
        "play_duration": random.randint(100, 300),  # 0-300 seconds
        "timestamp": int(time.time()),
        "rating": round(1 + (i * 0.5) % 4, 1)  # 1.0-5.0 rating
    }
    record_api_records.append(record)

print("📋 Generated 10 sample records:")
for i, record in enumerate(record_api_records, 1):
    print(f"   {i}. User {record['user_id']} {record['interaction_type']}ed {record['item_id']} (rating: {record['rating']})")

🔹 GENERATING SAMPLE INTERACTION DATA
📋 Generated 10 sample records:
   1. User luli played Wednesday (Season 1) (rating: 1.0)
   2. User wayne clicked Adolescence (Limited Series) (rating: 1.5)
   3. User chloe likeed Stranger Things 4 (rating: 2.0)
   4. User yaru shareed DAHMER: Monster: The Jeffrey Dahmer Story (rating: 2.5)
   5. User Yen played Bridgerton (Season 1) (rating: 3.0)
   6. User Vis clicked The Queen's Gambit (Limited Series) (rating: 3.5)
   7. User luli likeed Wednesday (Season 1) (rating: 4.0)
   8. User wayne shareed Adolescence (Limited Series) (rating: 4.5)
   9. User chloe played Stranger Things 4 (rating: 1.0)
   10. User yaru clicked DAHMER: Monster: The Jeffrey Dahmer Story (rating: 1.5)
   11. User Yen likeed Bridgerton (Season 1) (rating: 2.0)
   12. User Vis shareed The Queen's Gambit (Limited Series) (rating: 2.5)
   13. User luli played Wednesday (Season 1) (rating: 3.0)
   14. User wayne clicked Adolescence (Limited Series) (rating: 3.5)
   15. User chl

## 🔄 Convert to Rose Format

Let's convert our records to Rose's internal format:

In [14]:
# Convert records to Rose format
rose_records = convert_records_to_rose_format(record_api_records)

print("📋 Format Conversion:")
print("   Original format:")
print(f"   {record_api_records[0]}")
print("   " + "="*50)
print("   Rose format:")
print(f"   {rose_records[0]}")
print(f"\n✅ Converted {len(rose_records)} records to Rose format")


📋 Format Conversion:
   Original format:
   {'user_id': 'luli', 'item_id': 'Wednesday (Season 1)', 'item_type': 'video', 'interaction_type': 'play', 'play_duration': 192, 'timestamp': 1764659634, 'rating': 1.0}
   Rose format:
   {'user_id': {'str': 'luli'}, 'item_id': {'str': 'Wednesday (Season 1)'}, 'item_type': {'str': 'video'}, 'interaction_type': {'str': 'play'}, 'play_duration': {'int': 192}, 'timestamp': {'int': 1764659634}, 'rating': {'float': 1.0}}

✅ Converted 15 records to Rose format


## 📤 Ingest Records via Record API

Now let's send our records to the Rose API:

In [28]:
# Ingest records using Record API
try:
    print("📤 Ingesting records via Record API...")
    response = client.datasets.records.patch(interaction_dataset_id, rose_records)
    
    print(f"✅ Successfully ingested {len(record_api_records)} records!")
    print(f"   Response: {response}")
    
except RoseAPIError as e:
    print(f"❌ Failed to ingest records: {e.message}")
    print(f"   Error details: {e}")

print(f"\n🎉 Step 3 Complete! Ingested {len(record_api_records)} records using Record API")


📤 Ingesting records via Record API...
✅ Successfully ingested 15 records!
   Response: None

🎉 Step 3 Complete! Ingested 15 records using Record API


## ✅ Query results


In [27]:
# List all queries
try:
    print(f"Getting all queries from Pipeline : {pipeline_id}")
    queries = client.pipelines.list_queries(pipeline_id=pipeline_id)
    query_fields = ["avg_rating", "item_count", "play_duration_sum"]
    for q, f in zip(queries, query_fields):
        print("="*10 , q.query_id, "="*10)
        recommendations = client.recommendations.get(query_id=q.query_id, parameters={"size":10, "gte": "now-2d"})
        print(recommendations)
        # for i, rec in enumerate(recommendations.results.buckets, 1):
        #     print(f"  {i}. {rec.key_as_string} - {f} -  {rec.get_metric(f)}")
except RoseAPIError as e:
    print(f"❌ Failed to ingest records: {e.message}")
    print(f"   Error details: {e}")

Getting all queries from Pipeline : aAGYFsdwT5-17XKztWXPww
========== 2ab1923189176f9e152ec6c14bb69af6 ==========
results=AggregationResults(doc_count_error_upper_bound=0, sum_other_doc_count=0, buckets=[])
========== 9fd4afeea1aa80f59773c814d0196994 ==========
results=AggregationResults(doc_count_error_upper_bound=0, sum_other_doc_count=0, buckets=[])
========== fd3097abbd05e85c1793c087c13e6178 ==========
results=AggregationResults(doc_count_error_upper_bound=0, sum_other_doc_count=0, buckets=[])


In [37]:
def get_results(
        index_name,
        past_n_min: int,
        top_k: int,
    ):

        terms = [{"field": col} for col in ["item_id", "item_type"]]

        oss_query = {
            "size": 0,
            "query": {
                "range": {
                    "_start_timestamp": {"gte": f"now-{past_n_min}m"},
                }
            },
            "aggs": {
                "results": {
                    "multi_terms": {"terms": terms, "size": top_k, "order": {"score": "desc"}},
                    "aggs": {"score": {"sum": {"field": "_value"}}},
                }
            },
        }

        # Execute search query
        data = oss_client.search(
            index=index_name, body=oss_query
        )

        # Process results
        popular_items = (
            data.get("aggregations", {}).get("results", {}).get("buckets", [])
        )

        results = {}
        for item in popular_items[:top_k]:  # Ensure we don't exceed top_k
            key = (
                ":".join(str(k) for k in item["key"])
                if isinstance(item["key"], list)
                else str(item["key"])
            )
            score = item.get("score", {}).get("value", 0)
            results[key] = score
            print(f"Item: {key}", f"Value: {score}")

        
        return results